# MHE на реальных бэгах (распаршенные CSV)

Гоняет acados-MHE (модели и веса продового кодогена: `mhe_codegen.py::read_mhe_params`)
на CSV из `<sda_context>/logs/ceed/<bag>/parsed/` (продукт `bag_to_csv/to_csv.py`).

Зеркалим рантайм C++ (`MhePipeline`): горизонт N=500×20 мс, solve каждые SHIFT=50
сэмплов (раз в 1 с), vx-пороги per-model, между сегментами движения и после
неудачного solve — аналог `clear()` (re-warm-start солвера), Σ arrival cost и
прайор параметров при этом живут. Гейтов рантайма info_gain_ref / ρ̂ /
ParamUpdateGate в Python-цикле **нет** — это «чистый» MHE; clamp/floor Σ
(как в C++ `update()`) включаются пресетом `MACHINE_LIKE`; машинные σ и честные
CI досчитываются постфактум (`sigma_bands`, см. секцию треков).

Окружение: **только dev-контейнер** (acados_template + casadi стоят там).
Headless smoke (m11 / DynamicModel / 6 окон):
```bash
docker exec -e MPLBACKEND=Agg -e MHE_NB_SMOKE=1 -w /rep/scripts/control_scripts \
  <контейнер> python3 -c "import nbformat; nb=nbformat.read('mhe_test_rosbag.ipynb',as_version=4); \
exec(compile(chr(10).join(''.join(c.source) for c in nb.cells if c.cell_type=='code'),'nb','exec'),{'__name__':'__main__'})"
```
Без `MHE_NB_SMOKE` — полный прогон; `MHE_NB_BAG=random_dev` переключает бэг;
`MHE_NB_MACHINE=1` — пресет «как нода» (латчинг + max_iter 15 + kinematic θ0=0.1
+ clamp/floor Σ в arrival cost).
В headless-режиме все графики пишутся html-файлами в `GEN_ROOT`.

Родня: `control/libs/mpc/codegen/tests/mhe/mhe_test.py` (то же на синтетике),
`tests_utils/mhe_utils.py` (обвязка + общий шаг окна `solve_mhe_window`),
`tests_utils/mhe_rosbag_utils.py` (сегменты/раннер/треки/σ-ленты),
`bag_to_csv/{to_csv,check_dynamic_mhe_inputs}.py` (парсер и LS-референсы),
`.claude/docs/mhe.md`.


In [1]:
# --- bootstrap -------------------------------------------------------------
import gc  # noqa: F401  (нужен при ручной пересборке солверов, см. секцию про кэш)
import os
import sys
from pathlib import Path

import numpy as np
import yaml

try:  # интерактивно: автоперезагрузка правок mhe_rosbag_utils
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

IN_CONTAINER = Path("/rep").exists()
SDA_ROOT = Path("/rep") if IN_CONTAINER else Path.home() / "autotech/SDA"
DATA_ROOT = (Path("/sda_context") if IN_CONTAINER else Path.home() / "sda_context") / "logs/ceed"
CODEGEN_ROOT = SDA_ROOT / "control/libs/mpc/codegen"
for _p in (CODEGEN_ROOT, SDA_ROOT / "scripts/control_scripts",
           SDA_ROOT / "scripts/control_scripts/bag_to_csv"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import plotly.graph_objects as go  # noqa: E402
from plotly.subplots import make_subplots  # noqa: E402

SMOKE = os.environ.get("MHE_NB_SMOKE") == "1"

# --- ручки -----------------------------------------------------------------
BAG = os.environ.get("MHE_NB_BAG", "m11")   # m11 (шоссе) | random_dev (манёвры)
MODELS = ["DynamicModel", "KinematicModel"]
N = 500                          # горизонт: 500 x 20 мс = 10 с (прод)
SHIFT = 50                       # сдвиг окна = TIMELAP_CYCLES (solve раз в 1 с)
DT = 0.02
V_EPS = 2.0                      # общий порог vx [м/с] — как v_eps на машине/CIL
GAP_GUARD_S = 0.2                # разрыв сырых данных больше этого режет сегмент
MAX_SQP_ITER = 150               # как в mhe_test.py (прод C++: 15 + SQP-варм между solve)
ACCEPT_MAX_ITER = False          # status==2 (упёрся в max iter) считать годным
# Несовместимый кэш солверов (собран другой версией acados) get_or_build_solver
# распознаёт и пересобирает сам. Здесь — принудительная пересборка после
# правок модели: MHE_NB_FORCE_REBUILD=1
FORCE_REBUILD = os.environ.get("MHE_NB_FORCE_REBUILD") == "1"
MAX_WINDOWS = None               # None = все; число — для быстрых проб
RESET_PRIOR_BETWEEN_SEGMENTS = False  # True: каждый сегмент с чистого прайора (сравнение)
RUN_DELAY = False                # см. delay-секцию внизу
DELAY_ENGAGED_RANGES = []        # [(t_от, t_до), ...] в нормализованном времени
DECIM = 2                        # прореживание сырых трейсов; станет вязко — подними (SVG-рендер)
SAMPLING = "interp"              # interp | latch («последнее пришедшее» на тике — как нода)
MACHINE_LIKE = os.environ.get("MHE_NB_MACHINE", "0") == "1"  # или True руками

GEN_ROOT = (CODEGEN_ROOT / "build/mhe_rosbag").resolve()
GEN_ROOT.mkdir(parents=True, exist_ok=True)

if SMOKE:
    BAG, MODELS = "m11", ["DynamicModel"]
    MAX_WINDOWS, MAX_SQP_ITER, RUN_DELAY = 6, 50, False

# --- палитра и вывод фигур -------------------------------------------------
C_BLUE, C_ORANGE, C_AQUA, C_RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
C_GRAY = "#52514e"


def rgba(hex_color, alpha):
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


def show_fig(fig, name):
    """Интерактивно — inline; headless (exec без IPython) — html в GEN_ROOT."""
    try:
        get_ipython()
        fig.show()
    except NameError:
        out = GEN_ROOT / f"{name}_{BAG}.html"
        fig.write_html(str(out), include_plotlyjs="cdn")
        print(f"[fig] {out}")


# --- маппинг сигналов на модели --------------------------------------------
WHEELBASE, GR_NOM = 2.65, 14.0
MODEL_SPECS = {
    # u_cols/y_cols — имена батчей LogReaderV2; x0_func: (y0, u0, theta) -> state.
    # θ0 — физические старты; на машине иначе: kinematic стартует с GR=0.1
    # (верх yaml), dynamic — из PhysicalParams фабрики (~0.068 для ceed).
    # Σ0/q — std_state_aug и state_aug_noise рантайма (dynamic — машинный yaml,
    # kinematic/delay — C++-дефолты make_config).
    "DynamicModel": dict(
        u_cols=["vx", "angle"], y_cols=["ay", "om_z"],
        x0_func=lambda y0, u0, th: np.array([y0[1], 0.0]),
        theta0=np.array([0.068, 0.0]),
        sigma0_std=np.array([10.0, 10.0, 0.05, 0.05]),
        q_state=1e-7, q_param=1e-8,
        min_vx=3.5,
        param_names=["K_us [м]", "offset [рад]"],
        state_names=["wz [рад/с]", "vy_rear [м/с]"],
        meas_names=["a_lat [м/с²]", "wz [рад/с]"],
    ),
    "KinematicModel": dict(
        u_cols=["vx", "angle", "wheelbase"], y_cols=["yaw", "om_z"],
        x0_func=lambda y0, u0, th: np.array([y0[0]]),
        theta0=np.array([1.0 / GR_NOM, 0.0]),
        sigma0_std=np.array([1.0, 0.05, 0.05]),
        q_state=1e-7, q_param=np.array([6e-8, 5e-8]),  # C++ дефолт: GR 6e-8, offset 5e-8
        min_vx=0.0,
        param_names=["GR", "offset [рад]"],
        state_names=["ψ [рад]"],
        meas_names=["ψ [рад]", "wz [рад/с]"],
    ),
    "Delay": dict(
        # DelaySystemZ: z-координаты Паде-2 (z2 = tau/2 · x2 старой реализации),
        # наблюдение y = u - 2*z2 параметра не содержит; θ = [tau, offset] как раньше
        u_cols=["steer_req"], y_cols=["angle"],
        x0_func=lambda y0, u0, th: np.array([u0[0], 0.0]),
        theta0=np.array([0.15, 0.0]),
        sigma0_std=np.array([1.0, 1.0, 0.05, 0.05]),
        q_state=1e-6, q_param=np.array([5e-7, 1e-6]),  # C++ дефолт: tau 5e-7, offset 1e-6
        min_vx=0.0,
        param_names=["tau [с]", "offset [рад]"],
        state_names=["z1", "z2"],
        meas_names=["SWA [рад]"],
    ),
}

if MACHINE_LIKE:
    # максимум похожести на ноду: латчинг с его лагами, прод max_iter, старт
    # кинематики GR=0.1 (в yaml нет init_params), clamp/floor Σ в updater'е
    # (прокидывается в run_mhe_on_segments в ячейке прогона). Всё равно не 1:1 —
    # джиттер rosbag play, фаза окон/clear(), ParamUpdateGate на выходе mhe3.
    SAMPLING, MAX_SQP_ITER = "latch", 15
    MODEL_SPECS["KinematicModel"]["theta0"] = np.array([0.1, 0.0])
    print("MACHINE_LIKE: latch + max_iter 15 + kinematic θ0=0.1 + clamp/floor Σ")

with open(SDA_ROOT / "sda_config/control.param.yaml") as _f:
    CONTROL_PARAM_YAML = yaml.safe_load(_f)

BAG_DIR = DATA_ROOT / BAG / "parsed"
assert BAG_DIR.exists(), f"нет данных: {BAG_DIR}"
print(f"bag: {BAG_DIR}   контейнер: {IN_CONTAINER}   SMOKE: {SMOKE}")


ModuleNotFoundError: No module named 'yaml'

## Данные и конвенции

| csv | что берём | примечание |
|---|---|---|
| pose_aligned | vx (кузовной, twist), q0..q3 = (w,x,y,z) → yaw | om_z из pose НЕ берём (лаг цепочки локализации) |
| unbiased_imu | ay (боковое, REP-103), om_z (гироскоп), ax | синхронная пара ay+wz для DYNAMIC — как в ноде |
| steering | angle = SWA [рад], измеренный руль | rwa = angle/GR — только для sanity |
| control_req | steer_req [рад], acc_req/dec_req | steer_req — для DELAY |

Вход `steering` DYNAMIC/KINEMATIC — **измеренный** руль, не команда: лаг руля
смертелен для K (δK = g·δrwa/a_lat, см. mhe.md). Сэмплирование на сетке 20 мс:
`SAMPLING="interp"` — линейная интерполяция (лаг 0, сглаживание);
`"latch"` — ZOH «последнее пришедшее» по receive-времени, как латчит нода
(вместе с её эффективными лагами ~полпериода топика).


In [ ]:
from experiments.data_utils import LogReaderV2  # наш LogReaderV2 из GaussNewton;
# путь даёт editable-установка. Раньше было `from data_utils import ...` — оно
# резолвилось в тот же файл случайно, через каталог ноутбука в sys.path[0]

reader = LogReaderV2({
    "pose": BAG_DIR / "pose_aligned.csv",
    "imu": BAG_DIR / "unbiased_imu.csv",
    "steering": BAG_DIR / "steering.csv",
    "req": BAG_DIR / "control_req.csv",
})

# yaw: atan2 по сырым строкам -> unwrap (в pose q0=w, q1=x, q2=y, q3=z — см. to_csv.py)
_pose = reader.dfs["pose"]
_q = _pose[["q0", "q1", "q2", "q3"]].to_numpy()
_flips = int((np.sum(_q[1:] * _q[:-1], axis=1) < 0).sum())
assert _flips == 0, f"знаковые флипы кватернионов: {_flips} — unwrap по atan2 небезопасен"
_yaw = np.arctan2(2.0 * (_q[:, 0] * _q[:, 3] + _q[:, 1] * _q[:, 2]),
                  1.0 - 2.0 * (_q[:, 2] ** 2 + _q[:, 3] ** 2))
_pose["yaw"] = np.unwrap(_yaw)

for _col, _src in [("vx", "pose"), ("yaw", "pose"), ("ay", "imu"), ("om_z", "imu"),
                   ("ax", "imu"), ("angle", "steering"), ("rwa", "steering"),
                   ("steer_req", "req"), ("acc_req", "req"), ("dec_req", "req")]:
    reader.add_batch(_col, source=_src, use_jax_interp=False)
reader.process_all()
T1, T2 = reader.get_common_time_range()
print(f"общий интервал: [{T1:.2f}, {T2:.2f}] с ({T2 - T1:.1f} с)")


In [ ]:
# --- sanity: частоты, знаки, оси -------------------------------------------
for _name in ["vx", "ay", "om_z", "angle", "steer_req"]:
    _t = reader.get_time(_name)
    _d = np.diff(_t)
    print(f"{_name:10s} dt медиана {np.median(_d) * 1e3:5.1f} мс, max гэп {_d.max() * 1e3:6.0f} мс")

_swa_raw, _ = reader.get_raw_data("angle")
_rwa_raw, _ = reader.get_raw_data("rwa")
gr_eff = np.polyfit(_rwa_raw, _swa_raw, 1)[0]
_vx_raw, _ = reader.get_raw_data("vx")
print(f"GR_eff (SWA/RWA) = {gr_eff:.2f} (номинал {GR_NOM})")
print(f"vx: [{_vx_raw.min():.2f}, {_vx_raw.max():.2f}] м/с; "
      f"ниже {V_EPS}: {(_vx_raw < V_EPS).mean() * 100:.1f}%, ниже 3.5: {(_vx_raw < 3.5).mean() * 100:.1f}%")

# знаки: в левом повороте SWA>0, wz>0, ay>0 (REP-103)
_tt = np.arange(T1, T2, 0.05)
_f = reader.get_f_interp
_swa, _wz, _ay, _vx = _f("angle")(_tt), _f("om_z")(_tt), _f("ay")(_tt), _f("vx")(_tt)
_m = _vx > 3.0
_c1 = np.corrcoef(_swa[_m], _wz[_m])[0, 1]
_c2 = np.corrcoef(_ay[_m], (_vx * _wz)[_m])[0, 1]
print(f"corr(SWA, wz) = {_c1:.3f} (ожидаем > 0), corr(ay, vx*wz) = {_c2:.3f} (ожидаем > 0.8)")
assert _c2 > 0.8, "ось/знак ay подозрительны — проверь to_csv/бэг"

fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=["vx [м/с]", "SWA [рад]", "ay (боковое) [м/с²]",
                                    "wz [рад/с]", "a_x (продольное) [м/с²]",
                                    "запросы acc/dec [м/с²]"])
for _r, _name in enumerate(["vx", "angle", "ay", "om_z", "ax"], start=1):
    _v, _t = reader.get_raw_data(_name)
    fig.add_trace(go.Scatter(x=_t[::DECIM], y=_v[::DECIM], mode="lines",
                             line=dict(color=C_BLUE, width=1), showlegend=False),
                  row=_r, col=1)
for _name, _color in [("acc_req", C_BLUE), ("dec_req", C_ORANGE)]:
    _v, _t = reader.get_raw_data(_name)
    fig.add_trace(go.Scatter(x=_t[::DECIM], y=_v[::DECIM], mode="lines", name=_name,
                             line=dict(color=_color, width=1)), row=6, col=1)
fig.update_layout(template="plotly_white", height=6 * 170 + 130, width=1200,
                  font=dict(size=12, color="#0b0b0b"), hovermode="x unified",
                  title=f"{BAG}: обзор сигналов (децимация 1:{DECIM})",
                  legend=dict(orientation="v", x=1.01, xanchor="left",
                              y=1.0, yanchor="top"),
                  margin=dict(r=140))
fig.update_xaxes(title_text="t [с]", row=6, col=1)
show_fig(fig, "overview")


## Сетка, маски, сегменты

Сетка 20 мс строго внутри общего интервала (interp1d за краями экстраполирует —
не вылезать). Маска per-model: `vx > max(V_EPS, min_vx)` (машина: dynamic
3.5 м/с) минус гэп-гард. Непрерывные куски длиной ≥ N — сегменты; внутри — окна
`[i, i+N)` с шагом SHIFT. Всё короче N (10 с) выпадает — как на машине после
`clear()` MHE копит буфер заново.


In [ ]:
from mhe_rosbag_utils import (build_grid, find_segments, gap_guard_mask,
                                          make_window_starts)

t_grid = build_grid(T1, T2, DT)
Ng = len(t_grid)


def _sample(name):
    if SAMPLING == "latch":  # ZOH по receive-времени — латчинг ноды вместе с его лагами
        _v, _ts = reader.get_raw_data(name)
        _idx = (np.searchsorted(_ts, t_grid, side="right") - 1).clip(0, len(_ts) - 1)
        return _v[_idx]
    return reader.get_f_interp(name)(t_grid)


SIGNALS = {n: _sample(n) for n in
           ["vx", "yaw", "ay", "om_z", "angle", "steer_req"]}
SIGNALS["wheelbase"] = np.full(Ng, WHEELBASE)
print(f"сэмплирование: {SAMPLING}")

gap = gap_guard_mask(t_grid, [reader.get_time(n) for n in ["vx", "ay", "om_z", "angle"]],
                     GAP_GUARD_S)

WINDOWS, SEGMENTS = {}, {}
for _m in MODELS:
    _mask = (SIGNALS["vx"] > max(V_EPS, MODEL_SPECS[_m]["min_vx"])) & ~gap
    SEGMENTS[_m] = find_segments(_mask, min_len=N)
    WINDOWS[_m] = make_window_starts(SEGMENTS[_m], N, SHIFT)
    _nw = sum(len(sw.window_starts) for sw in WINDOWS[_m])
    _cov = sum(s1 - s0 for s0, s1 in SEGMENTS[_m]) * DT
    print(f"{_m}: сегментов {len(SEGMENTS[_m])}, окон {_nw}, покрыто {_cov:.0f} с из {Ng * DT:.0f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_grid[::DECIM], y=SIGNALS["vx"][::DECIM], mode="lines",
                         name="vx", line=dict(color=C_BLUE, width=1.5)))
for _s0, _s1 in SEGMENTS[MODELS[0]]:
    fig.add_vrect(x0=t_grid[_s0], x1=t_grid[_s1 - 1], fillcolor=rgba(C_AQUA, 0.12),
                  line_width=0, layer="below")
fig.add_hline(y=max(V_EPS, MODEL_SPECS[MODELS[0]]["min_vx"]),
              line=dict(color=C_GRAY, width=1, dash="dot"))
fig.update_layout(template="plotly_white", height=420, width=1200,
                  font=dict(size=12, color="#0b0b0b"), hovermode="x",
                  title=f"сегменты {MODELS[0]} (заливка) и vx-порог (пунктир)",
                  yaxis_title="vx [м/с]", xaxis_title="t [с]")
show_fig(fig, "segments")


## Солверы (кодоген + кэш)

Кэш в `codegen/build/mhe_rosbag/<name>_N<N>_it<iter>` (gitignored через `build/`):
если json + .so на месте — reload без пересборки (`FORCE_REBUILD=True` — пересобрать).

Гигиена ядра:
- солверы держать **только** в dict `SOLVERS` и не выводить последним выражением
  ячейки (Out[] IPython пиннит ссылку навечно → dlopen по тому же пути вернёт
  старый handle вместо пересобранного);
- перед пересборкой того же конфига: `SOLVERS.pop(имя, None); gc.collect()`;
  надёжнее — рестарт ядра;
- смена N / MAX_SQP_ITER = новая подпапка кэша, старые можно удалять целиком.


In [ ]:
from mhe_codegen import MHE_YAML_KEYS, read_mhe_params
from mhe_rosbag_utils import get_or_build_solver

SOLVERS, MHE_MODELS, MHE_PARAMS = {}, {}, {}
for _m in MODELS + (["Delay"] if RUN_DELAY else []):
    _params, _system = read_mhe_params(CONTROL_PARAM_YAML, _m)
    _params.mhe_horizont = N  # до конструктора генератора; tf/N_horizon возьмутся отсюда
    _model, _solver = get_or_build_solver(
        _system, _params, GEN_ROOT, model_name=f"{_m.lower()}_rosbag",
        max_sqp_iter=MAX_SQP_ITER, force_rebuild=FORCE_REBUILD)
    SOLVERS[_m], MHE_MODELS[_m], MHE_PARAMS[_m] = _solver, _model, _params
print(f"готово: {list(SOLVERS)}")


## Прогон

Ядро окна — общий `mhe_utils.solve_mhe_window` (то же, что в синтетическом
тесте); поверх — зеркало машины: re-warm-start на границах сегментов и после
фейла solve, Σ и θ-прайор живут сквозь сегменты
(`RESET_PRIOR_BETWEEN_SEGMENTS=True` — режим сравнения). Вместе с оценкой
сохраняется `obs_est` = h(x_est, θ_est, u) — предсказание измерений моделью
вдоль оценённой траектории (кормит график «измерения vs модель» и расчёт ρ̂).


In [ ]:
from mhe_rosbag_utils import run_mhe_on_segments, summarize_results

MHE_SECTION = CONTROL_PARAM_YAML["/**"]["ros__parameters"]["control"]["mhe_params"]
RESULTS = {}
for _m in MODELS:
    _spec = MODEL_SPECS[_m]
    _U = np.column_stack([SIGNALS[c] for c in _spec["u_cols"]])
    _Y = np.column_stack([SIGNALS[c] for c in _spec["y_cols"]])
    _sec = MHE_SECTION[MHE_YAML_KEYS[_m]]
    print(f"--- {_m}: U{_U.shape} Y{_Y.shape}")
    RESULTS[_m] = run_mhe_on_segments(
        MHE_MODELS[_m], SOLVERS[_m], MHE_PARAMS[_m], t_grid, _U, _Y, WINDOWS[_m],
        x0_func=_spec["x0_func"], theta0=_spec["theta0"],
        sigma0=np.diag(_spec["sigma0_std"] ** 2), shift=SHIFT,
        q_state=_spec["q_state"], q_param=_spec["q_param"],
        r_inv=MHE_PARAMS[_m].measurements_residual_r,
        # MACHINE_LIKE: clamp/floor Σ внутри updater'а — как C++ update();
        # иначе «чистый» MHE без ограничителей (ленты raw/eff не искажаются)
        sigma_max_std=_spec["sigma0_std"] if MACHINE_LIKE else None,
        sigma_min_ratio=(float(_sec.get("sigma_min_ratio", 1e-3))
                         if MACHINE_LIKE else 0.0),
        reset_prior_between_segments=RESET_PRIOR_BETWEEN_SEGMENTS,
        accept_max_iter=ACCEPT_MAX_ITER, max_windows=MAX_WINDOWS)
    summarize_results(_m, RESULTS[_m], _spec["param_names"])


In [ ]:
# --- офлайн LS-референс (квазистационарная кинематика+градиент) -------------
# формулы и фиттеры — в bag_to_csv/check_dynamic_mhe_inputs.py (единый источник)
from check_dynamic_mhe_inputs import (ls_reference_dynamic, ls_reference_kinematic,
                                      quasi_stationary_mask)

qs = quasi_stationary_mask(SIGNALS["vx"], SIGNALS["angle"], DT, vmin=3.5)
print(f"квазистационарных сэмплов: {int(qs.sum())} ({qs.mean() * 100:.1f}%)")

LS_REF = {}
_th_d, _info_d = ls_reference_dynamic(SIGNALS["vx"], SIGNALS["angle"], SIGNALS["om_z"],
                                      qs, gr=GR_NOM, wheelbase=WHEELBASE)
if _th_d is not None:
    LS_REF["DynamicModel"] = _th_d
    print(f"LS dynamic:   K = {_th_d[0]:.4f} м (ceed-ожидание ~0.068), "
          f"offset = {np.degrees(_th_d[1]):.3f}°   (n={_info_d['n']}, cond={_info_d['cond']:.0f})")
else:
    print(f"LS dynamic: {_info_d}")

# GR_eps фита поглощает недостаточную поворачиваемость — на высокой скорости
# ожидаемо ниже 1/GR; сравнивать с kinematic-MHE, а не с номиналом
_th_k, _info_k = ls_reference_kinematic(SIGNALS["vx"], SIGNALS["angle"], SIGNALS["om_z"],
                                        qs, wheelbase=WHEELBASE)
if _th_k is not None:
    LS_REF["KinematicModel"] = _th_k
    print(f"LS kinematic: GR_eps = {_th_k[0]:.4f} (номинал {1 / GR_NOM:.4f}), "
          f"offset = {np.degrees(_th_k[1]):.3f}°   (n={_info_k['n']})")
else:
    print(f"LS kinematic: {_info_k}")


## Треки параметров и ленты неопределённости

- **синяя лента** — сырая ±σ из Σ_θθ arrival cost: веса W нефизичны, масштаб
  условный; годится только как относительный индикатор информации;
- **жёлтая лента** — «σ как на машине»: порядок как в C++ — clamp/floor режут
  саму Σ (внутри `update()`), и только **потом** диагностика умножает на ρ̂
  (`update_diagnostics`), т.е. `ρ̂·clip(σ_raw, ratio·σ0, σ0)`; на полу машина
  публикует ρ̂·ratio·σ0, а не ratio·σ0. Воспроизводит std_devs топика mhe3
  (покомпонентный clip ≈ спектральный clamp; при `MACHINE_LIKE` clamp/floor
  уже внутри updater'а и clip здесь no-op). ρ̂ — апостериорный масштаб единицы
  веса (ρ̂² = ΣrᵀWr/dof, EMA α=0.1, зажим `residual_scale_bounds`); на реальных
  данных прижата к полу 0.01 — отсюда «на порядки уже» сырой;
- **маджента-пунктир** — ±σ_eff: честный локальный CI = ρ̂·σ·√n_corr,
  HAC-поправка на автокорреляцию невязки (dof считает 1000 измерений окна
  независимыми, реально независимых ~N/n_corr);
- **серые точки** — скользящий разброс самой оценки (окно 30 solve) —
  маршрутный интервал; систематику (уклон, лаги) не покрывает никакая
  ковариация, честно только это.

Оранжевый пунктир — LS-референс, чёрные точечные — границы yaml, красные
vlines — нерешённые окна, зелёная заливка — сегменты.

**Математика**: линеаризованно θ̂ − θ° = (JᵀWJ)⁻¹JᵀWe ⇒ сэндвич
Cov(θ̂) = (JᵀWJ)⁻¹(JᵀW·C_e·WJ)(JᵀWJ)⁻¹. Белый шум с C_e = σ₀²W⁻¹ даёт
ρ̂·σ_raw (ρ̂² = êᵀWê/dof — оценка σ₀²) — точный CI; стационарная
автокорреляция — σ_eff = ρ̂·σ_raw·√n_corr (Ньюи–Уэст) — асимптотический CI
**при верной модели**; смещение модели b(t) двигает E[θ̂] на (JᵀWJ)⁻¹JᵀW·b,
в невязке почти не видно и никакой σ не покрывается. Поэтому σ_eff — нижняя
граница (мала законно: arrival cost усредняет ~50+ с данных), скользящий
разброс — рабочая верхняя; их отношение — мера маршрутной систематики.

**Почему сравнение с mhe3 из реплея не 1:1**: нода латчит сообщения на тике
(эффективный лаг ~полпериода топика + джиттер rosbag play — включается
`SAMPLING="latch"`), у ноды `max_iter=15` (тут 150 — пресет `MACHINE_LIKE`),
mhe3 фильтруется `ParamUpdateGate` (прижатое к границе не публикуется), фаза
окон/clear() другая, kinematic стартует с GR=0.1.


In [ ]:
from mhe_rosbag_utils import plot_param_tracks, sigma_bands

MHE_SECTION = CONTROL_PARAM_YAML["/**"]["ros__parameters"]["control"]["mhe_params"]
BANDS = {}
for _m in MODELS:
    _spec = MODEL_SPECS[_m]
    _sec = MHE_SECTION[MHE_YAML_KEYS[_m]]
    # нет residual_scale_bounds в yaml == ρ̂ ≡ 1 в C++ (например delay)
    _rho_b = (tuple(_sec["residual_scale_bounds"])
              if "residual_scale_bounds" in _sec else (1.0, 1.0))
    BANDS[_m] = sigma_bands(
        RESULTS[_m], MHE_PARAMS[_m].measurements_residual_r, _spec["sigma0_std"],
        rho_bounds=_rho_b, sigma_min_ratio=float(_sec.get("sigma_min_ratio", 1e-3)))
    print(f"{_m}: ρ̂ сырая p50={np.median(BANDS[_m]['rho_raw']):.4g} (зажим {_rho_b}), "
          f"n_corr p50={np.median(BANDS[_m]['n_corr']):.0f}")
    fig = plot_param_tracks(
        RESULTS[_m], _spec["param_names"],
        ls_ref=LS_REF.get(_m),
        bounds=(_sec["param_bounds_lower"], _sec["param_bounds_upper"]),
        theta0=_spec["theta0"],
        segments_time=[(t_grid[s0], t_grid[s1 - 1]) for s0, s1 in SEGMENTS[_m]],
        title=f"{BAG} / {_m}", bands=BANDS[_m])
    show_fig(fig, f"tracks_{_m}")


## Слайдер по окнам

Ряды: оценённые состояния; по подграфику на каждый канал измерений (измерение
vs выход observation-модели h(x_est, θ_est, u)); параметры ±σ. Выбранное окно
рисуется жирным оверлеем **полной длины N** и серой заливкой его диапазона —
при сдвиге SHIFT < N оверлей залезает на соседние окна, перекрытие видно.
Берём до 60 подряд идущих окон одного сегмента (полный прогон html не потянет).


In [ ]:
if not SMOKE and MODELS:
    from tests_utils.mhe_utils import plot_mhe_slider_lines

    _m = MODELS[0]
    _solved = [w for w in RESULTS[_m] if w.res is not None]
    _chain = []
    for _w in _solved:
        if _chain and (_w.seg_id != _chain[-1].seg_id
                       or _w.i_start - _chain[-1].i_start != SHIFT):
            if len(_chain) >= 40:
                break
            _chain = [_w]
        else:
            _chain.append(_w)
        if len(_chain) >= 60:
            break
    if len(_chain) >= 3:
        _spec = MODEL_SPECS[_m]
        fig = plot_mhe_slider_lines(
            [w.res for w in _chain], overlap=N - SHIFT,
            initial_params=_spec["theta0"],
            state_names=_spec["state_names"], meas_names=_spec["meas_names"],
            param_names=_spec["param_names"], figsize=(1300, 950))
        show_fig(fig, f"slider_{_m}")
        print(f"окон в слайдере: {len(_chain)}")
    else:
        print("непрерывной цепочки решённых окон не нашлось")


## Delay-MHE (выключено по умолчанию)

Наблюдение DelayOffset — feed-through от команды: на участках ручного руления
(команда не связана с рулём) оценка τ — мусор. Прокси `|steer_req|>0` не
работает (команда ненулевая почти всегда), поэтому engaged определяем по
согласию команды и факта (`|steer_req − angle| < 0.35 рад` с выдержкой ≥ 2 с),
при необходимости пересечь с ручными интервалами `DELAY_ENGAGED_RANGES`.
Включение: `RUN_DELAY=True` в конфиге (и перезапустить ячейку солверов).


In [ ]:
from scipy.ndimage import uniform_filter1d

_agree = np.abs(SIGNALS["steer_req"] - SIGNALS["angle"]) < 0.35
eng = uniform_filter1d(_agree.astype(float), size=int(2.0 / DT), mode="nearest") > 0.999
if DELAY_ENGAGED_RANGES:
    _manual = np.zeros(Ng, dtype=bool)
    for _ta, _tb in DELAY_ENGAGED_RANGES:
        _manual |= (t_grid >= _ta) & (t_grid <= _tb)
    eng &= _manual
print(f"engaged-прокси: {eng.mean() * 100:.1f}% времени")

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_grid[::DECIM], y=SIGNALS["angle"][::DECIM], mode="lines",
                         name="SWA факт", line=dict(color=C_BLUE, width=1.5)))
fig.add_trace(go.Scatter(x=t_grid[::DECIM], y=SIGNALS["steer_req"][::DECIM], mode="lines",
                         name="SWA команда", line=dict(color=C_ORANGE, width=1.5)))
for _s0, _s1 in find_segments(eng, min_len=int(2.0 / DT)):
    fig.add_vrect(x0=t_grid[_s0], x1=t_grid[_s1 - 1], fillcolor=rgba(C_AQUA, 0.10),
                  line_width=0, layer="below")
fig.update_layout(template="plotly_white", height=420, width=1200,
                  font=dict(size=12, color="#0b0b0b"), hovermode="x unified",
                  title="факт vs команда руля; заливка — engaged-прокси",
                  yaxis_title="[рад]", xaxis_title="t [с]",
                  legend=dict(orientation="v", x=1.01, xanchor="left",
                              y=1.0, yanchor="top"),
                  margin=dict(r=150))
show_fig(fig, "delay_overlay")

if RUN_DELAY:
    _spec = MODEL_SPECS["Delay"]
    _mask = (SIGNALS["vx"] > V_EPS) & eng & ~gap
    SEGMENTS["Delay"] = find_segments(_mask, min_len=N)
    WINDOWS["Delay"] = make_window_starts(SEGMENTS["Delay"], N, SHIFT)
    print(f"Delay: сегментов {len(SEGMENTS['Delay'])}, "
          f"окон {sum(len(sw.window_starts) for sw in WINDOWS['Delay'])}")
    _U = np.column_stack([SIGNALS[c] for c in _spec["u_cols"]])
    _Y = np.column_stack([SIGNALS[c] for c in _spec["y_cols"]])
    _sec = MHE_SECTION[MHE_YAML_KEYS["Delay"]]
    RESULTS["Delay"] = run_mhe_on_segments(
        MHE_MODELS["Delay"], SOLVERS["Delay"], MHE_PARAMS["Delay"], t_grid, _U, _Y,
        WINDOWS["Delay"], x0_func=_spec["x0_func"], theta0=_spec["theta0"],
        sigma0=np.diag(_spec["sigma0_std"] ** 2), shift=SHIFT,
        q_state=_spec["q_state"], q_param=_spec["q_param"],
        r_inv=MHE_PARAMS["Delay"].measurements_residual_r,
        sigma_max_std=_spec["sigma0_std"] if MACHINE_LIKE else None,
        sigma_min_ratio=(float(_sec.get("sigma_min_ratio", 1e-3))
                         if MACHINE_LIKE else 0.0),
        accept_max_iter=ACCEPT_MAX_ITER, max_windows=MAX_WINDOWS)
    summarize_results("Delay", RESULTS["Delay"], _spec["param_names"])
    _rho_b = (tuple(_sec["residual_scale_bounds"])
              if "residual_scale_bounds" in _sec else (1.0, 1.0))
    _bands_d = sigma_bands(
        RESULTS["Delay"], MHE_PARAMS["Delay"].measurements_residual_r,
        _spec["sigma0_std"], rho_bounds=_rho_b,
        sigma_min_ratio=float(_sec.get("sigma_min_ratio", 1e-3)))
    fig = plot_param_tracks(
        RESULTS["Delay"], _spec["param_names"],
        bounds=(_sec["param_bounds_lower"], _sec["param_bounds_upper"]),
        theta0=_spec["theta0"],
        segments_time=[(t_grid[s0], t_grid[s1 - 1]) for s0, s1 in SEGMENTS["Delay"]],
        title=f"{BAG} / Delay", bands=_bands_d)
    show_fig(fig, "tracks_Delay")


In [ ]:
# --- финальная сверка (работает и в SMOKE — это и есть headless-проверка) ---
print("=" * 64)
_active = list(MODELS) + (["Delay"] if RUN_DELAY and "Delay" in RESULTS else [])
for _m in _active:
    _spec = MODEL_SPECS[_m]
    _nt = len(_spec["param_names"])
    _solved = [w for w in RESULTS[_m] if w.res is not None]
    assert _solved, f"{_m}: ни одного решённого окна"
    _sec = CONTROL_PARAM_YAML["/**"]["ros__parameters"]["control"]["mhe_params"][MHE_YAML_KEYS[_m]]
    _lo = np.array(_sec["param_bounds_lower"])
    _hi = np.array(_sec["param_bounds_upper"])
    _th = _solved[-1].res.param_est
    assert np.all(_th >= _lo - 1e-9) and np.all(_th <= _hi + 1e-9), \
        f"{_m}: θ вне границ yaml: {_th}"
    _cov = _solved[-1].res.cov_matrix.reshape(_nt, _nt)
    assert np.all(np.isfinite(_cov)), f"{_m}: не-конечные значения в Σ"
    _stdF = np.sqrt(np.maximum(np.diag(_cov), 0))
    _std0 = _spec["sigma0_std"][-_nt:]
    # в чистом режиме clamp_covariance не применяется — σ невозбуждённого
    # параметра может чуть превышать σ0 за счёт Q-инфляции (норма; под
    # MACHINE_LIKE clamp/floor активны и σ_raw зажата в [ratio·σ0, σ0]);
    # ловим взрыв и полную глухоту
    assert np.all(_stdF < 3 * _std0), f"{_m}: σ взорвалась: {_stdF} vs σ0 {_std0}"
    assert np.min(_stdF / _std0) < 0.9, \
        f"{_m}: ни один параметр не получил информации: {_stdF} vs σ0 {_std0}"
    # obs_est обязан заполняться (кормит слайдер и расчёт ρ̂)
    assert _solved[-1].res.obs_est is not None and \
        _solved[-1].res.obs_est.shape == (N, len(_spec["meas_names"])), \
        f"{_m}: obs_est не заполнен/не той формы"
    if _m in BANDS and BANDS[_m]:
        assert np.all(np.isfinite(BANDS[_m]["sigma_machine"])), f"{_m}: NaN в sigma_machine"
    print(f"{_m}: OK  θ={np.round(_th, 5)}  σ={np.round(_stdF, 6)}  "
          f"окон {len(_solved)}/{len(RESULTS[_m])}")
print("ВСЁ OK")
